In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
data = gpd.read_file('distritos-peru@bogota-laburbano.geojson',encoding='utf8')[['nombdep','nombprov','nombdist']]
data = data.groupby(by=['nombdep','nombprov','nombdist']).size().reset_index()

deps = data['nombdep'].unique()
peru = {}
for dep in deps:
    temp_prov = {}
    provs = data[data['nombdep']==dep]['nombprov'].unique()
    for prov in provs:
        temp_prov.update({prov:[]})
        dists = data[(data['nombdep']==dep)&(data['nombprov']==prov)]['nombdist']
        for dist in dists:
            temp_prov[prov] += [dist]
    peru.update({dep:temp_prov})
    
noticias = pd.read_excel('nueva_base.xlsx')
noticias

,disease,date,title,body,tokens,domain
0,antrax,2010-08-29,Alarma en camal de Chincha por ántrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo
1,antrax,2012-01-12,Detectan dos casos de ántrax en Sama Inclán,(María LLayque).- La muerte de dos terneras (c...,mari llayqu muert dos terner cri hembr vac rep...,diariocorreo
2,antrax,2014-02-27,Tacna: Senasa descarta que ántrax hallada en o...,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp
3,antrax,2014-02-26,Senasa vacuna a ovinos y caprinos en Poquera p...,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo
4,antrax,2014-02-25,Confirman la presencia de ántrax en una oveja,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio
...,...,...,...,...,...,...
10213,zika,2024-04-28,Lucha contra el dengue en Piura: Minsa y Dires...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina
10214,zika,2024-05-08,Minsa incorpora ficha técnica del preparado fa...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina
10215,zika,2024-05-27,Aumentan a quince los casos de Guillain Barré ...,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo
10216,zika,2024-07-09,Piura: De seis piuranos solo uno recibe agua d...,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30


In [3]:
from tqdm import tqdm
tqdm.pandas()

def get_region(title:str,body:str):
    for dep, provs in peru.items():
        if f" {dep.lower()} " in title.lower() or f" {dep.lower()} " in body.lower():
            return dep
        
    for dep, provs in peru.items():   
        for prov, dists in provs.items():
            if f" {prov.lower()} " in title.lower() or f" {prov.lower()} " in body.lower():
                return dep
    
    for dep, provs in peru.items():   
        for prov, dists in provs.items():        
            for dist in dists:
                if f" {dist.lower()} " in title.lower() or f" {dist.lower()} " in body.lower():
                    return dep 
    return None

In [4]:
noticias['region'] = noticias.progress_apply(lambda x: get_region(x['title'],x['body']),axis=1)
noticias

100%|██████████| 10218/10218 [00:56<00:00, 180.88it/s]


,disease,date,title,body,tokens,domain,region
0,antrax,2010-08-29,Alarma en camal de Chincha por ántrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo,ICA
1,antrax,2012-01-12,Detectan dos casos de ántrax en Sama Inclán,(María LLayque).- La muerte de dos terneras (c...,mari llayqu muert dos terner cri hembr vac rep...,diariocorreo,TACNA
2,antrax,2014-02-27,Tacna: Senasa descarta que ántrax hallada en o...,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp,TACNA
3,antrax,2014-02-26,Senasa vacuna a ovinos y caprinos en Poquera p...,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo,TACNA
4,antrax,2014-02-25,Confirman la presencia de ántrax en una oveja,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio,TACNA
...,...,...,...,...,...,...,...
10213,zika,2024-04-28,Lucha contra el dengue en Piura: Minsa y Dires...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina,PIURA
10214,zika,2024-05-08,Minsa incorpora ficha técnica del preparado fa...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina,LA LIBERTAD
10215,zika,2024-05-27,Aumentan a quince los casos de Guillain Barré ...,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo,PIURA
10216,zika,2024-07-09,Piura: De seis piuranos solo uno recibe agua d...,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30,PIURA


In [ ]:
peru.